In [1]:
! pip install pdfplumber

In [10]:
import pdfplumber
import re
import pandas as pd
import glob

FOOTER_RE = re.compile(
    r'^https://rera\.tn\.gov\.in\S*.*$|^\d{1,2}/\d{1,2}/\d{2,4},.*::\s*TNRERA\s*::.*$',
    re.MULTILINE
)

def get_full_text(path):
    with pdfplumber.open(path) as pdf:
        raw = "\n".join(page.extract_text() or "" for page in pdf.pages)
    return FOOTER_RE.sub("", raw)

def safe_search(pattern, text, group=1, flags=re.I):
    m = re.search(pattern, text, flags)
    return m.group(group).strip() if m else None

def extract_project_header(page):
    words = page.extract_words()
    name_words = [w for w in words if w["text"] == "Name" and
                  any(w2["text"] == "Project" and abs(w2["top"] - w["top"]) < 2 for w2 in words)]
    if not name_words:
        return {}
    label_top = name_words[0]["top"]

    def x0_of(*phrase, top):
        for i in range(len(words) - len(phrase) + 1):
            chunk = words[i:i + len(phrase)]
            if [c["text"] for c in chunk] == list(phrase) and abs(chunk[0]["top"] - top) < 2:
                return chunk[0]["x0"]
        return None

    col_starts = {
        "Project_Name": x0_of("Project", "Name", top=label_top),
        "Project_Details": x0_of("Project", "Details", top=label_top),
        "Type_of_Building": x0_of("Type", "of", top=label_top),
        "Usage": x0_of("Usage", top=label_top),
    }
    col_starts = {k: v for k, v in col_starts.items() if v is not None}
    ordered = sorted(col_starts.items(), key=lambda kv: kv[1])

    end_words = [w for w in words if w["text"].startswith("Extent") and w["top"] > label_top]
    end_top = end_words[0]["top"] if end_words else label_top + 400

    bucket = {name: [] for name, _ in ordered}
    for w in words:
        if label_top + 2 < w["top"] < end_top:
            col = max((n for n, x0 in ordered if x0 <= w["x0"] + 3),key=lambda n: col_starts[n],default=ordered[0][0])
            bucket[col].append(w)

    result = {}
    for name, ws in bucket.items():
        ws.sort(key=lambda w: (round(w["top"], 1), w["x0"]))
        result[name] = " ".join(w["text"] for w in ws).strip()
    return result

def extract_fields(path):
    text = get_full_text(path)
    d = {"Source_File": path.split("/")[-1]}

    with pdfplumber.open(path) as pdf:
        d.update(extract_project_header(pdf.pages[0]))

    d["Site_Extent_Sqm"] = safe_search(r'Site Extent \(Sq\.m\)\s*:[^\n]*\n+([\d.]+)', text)
    d["Total_Dwelling_Units"] = safe_search(r'Phases\s*/\s*Villas\s*:[^\n]*\n(\d+)', text)
    d["Stage_of_Construction"] = safe_search(r'\b(Under Construction|Not Yet Started|Completed)\b', text)
    d["Project_Completion_Date"] = safe_search(
        r'Project Completion Date\s*:.*?\n.*?(\d{2}/\d{2}/\d{4})', text, flags=re.I | re.S
    )

    m = re.search(r'Latitude\s*:\s*Longitude\s*:\s*\n+\s*([\d.]+)\s+([\d.]+)', text, re.I)
    if m:
        d["Latitude"], d["Longitude"] = m.groups()

    m = re.search(
        r'No\. of Covered Car Parking\s*:\s*No\. of Open Car Parking\s*:\s*No\. of Visitor Car Parking\s*:\s*\n\s*(\d+)\s+(\d+)\s+(\d+)',
        text, re.I
    )
    if m:
        d["Covered_Parking"], d["Open_Parking"], d["Visitor_Parking"] = m.groups()

    m = re.search(
        r'Land Cost \(Market Value\)\s*:\s*Construction Cost\s*:\s*Other Cost\s*:\s*Total Project Cost\s*:\s*\n\s*([\d.]+)\s+([\d.]+)\s+(-|[\d.]+)\s+([\d.]+)',
        text, re.I
    )
    if m:
        d["Land_Cost"], d["Construction_Cost"], d["Other_Cost"], d["Total_Project_Cost"] = m.groups()

    m = re.search(
        r'<\s*60 Sq\.M \(LIG Residential\)\s*:\s*>\s*60 Sq\.M \(Other Residential\)\s*:\s*Commercial\s*:\s*Other Uses.*?\n\s*(-|[\d.]+)\s*(?:Sq\.m)?\s+(-|[\d.]+)\s*(?:Sq\.m)?\s+(-|[\d.]+)(?:\s*Sq\.m)?',
        text, re.I | re.S
    )
    if m:
        d["FSI_LIG_Residential"], d["FSI_Other_Residential"], d["FSI_Commercial"] = m.groups()

    pd_text = d.get("Project_Details", "") or ""
    d["Heights_m"] = [float(x) for x in re.findall(r'Height[^)]*?(\d+\.?\d*)\s*m', pd_text, re.I)]
    d["Floor_mentions"] = parse_floor_structures(pd_text)
    d["Districts"] = list(dict.fromkeys(re.findall(r'District\s*:\s*([A-Za-z ]+?)\s*\nPincode', text, re.I)))

    return d

CORE_FIELDS = [
    "Project_Name", "Site_Extent_Sqm", "Total_Dwelling_Units",
    "Latitude", "Longitude", "Land_Cost", "Construction_Cost",
    "Total_Project_Cost", "FSI_Other_Residential",
]

def run_batch(files):
    rows, failures = [], []
    for path in files:
        try:
            d = extract_fields(path)
            missing = [f for f in CORE_FIELDS if not d.get(f)]
            d["_missing_fields"] = ", ".join(missing) if missing else ""
            rows.append(d)
        except Exception as e:
            failures.append({"Source_File": path.split("/")[-1], "Error": str(e)})
    return pd.DataFrame(rows), pd.DataFrame(failures)

In [11]:
import re

NUMBER_WORDS = {
    "one": 1, "a": 1, "an": 1, "two": 2, "three": 3, "four": 4,
    "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
    "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14,
    "fifteen": 15, "sixteen": 16, "seventeen": 17, "eighteen": 18,
    "nineteen": 19, "twenty": 20,
    "first": 1, "second": 2, "third": 3, "fourth": 4, "fifth": 5,
    "sixth": 6, "seventh": 7, "eighth": 8, "ninth": 9, "tenth": 10,
    "eleventh": 11, "twelfth": 12, "thirteenth": 13, "fourteenth": 14,
    "fifteenth": 15, "sixteenth": 16, "seventeenth": 17,
    "eighteenth": 18, "nineteenth": 19, "twentieth": 20,
}

BASE_LEVEL_PATTERNS = [
    ("Combined Basement + Ground", r"\bcombined\s+(?:double\s+)?basement\s*\+\s*ground\b"),
    ("Basement", r"\bbasement\b"),
    ("Stilt", r"\bstilt\b|(?<!\w)s(?=\s*\+)"),
    ("Podium", r"\bpodium\b"),
    ("Ground", r"\bground\b|(?<!\w)g(?=\s*\+)"),
]
BLOCK_RE = re.compile(r"(?P<label>\bBlock\s+[^:\n]+):", re.IGNORECASE)


def _number_value(token):
    token = token.strip().casefold()
    return NUMBER_WORDS[token] if token in NUMBER_WORDS else float(token) if "." in token else int(token)


def _split_floor_blocks(text):
    matches = list(BLOCK_RE.finditer(text))
    if not matches:
        return [(None, text)]
    blocks = []
    for index, match in enumerate(matches):
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        label = re.sub(r"\s+to\s+", "-", match.group("label").strip(), flags=re.IGNORECASE)
        blocks.append((re.sub(r"\s+", " ", label), text[match.end():end]))
    return blocks


def _parse_floor_block(block_label, block_text):
    base_level = next(
        (label for label, pattern in BASE_LEVEL_PATTERNS
         if re.search(pattern, block_text, re.IGNORECASE)),
        None,
    )
    words = "|".join(sorted(NUMBER_WORDS, key=len, reverse=True))
    candidates = []
    count_re = re.compile(
        rf"\b(?P<count>\d+(?:\.\d+)?|{words})\s*(?:upper\s+)?floors?\b",
        re.IGNORECASE,
    )
    ordinal_re = re.compile(
        r"\b(?:\d+(?:st|nd|rd|th)|first|second|third|fourth|fifth|sixth|"
        r"seventh|eighth|ninth|tenth|eleventh|twelfth|thirteenth|"
        r"fourteenth|fifteenth|sixteenth|seventeenth|eighteenth|"
        r"nineteenth|twentieth)\s+floor\b", re.IGNORECASE,
    )
    named_re = re.compile(
        r"\b(?:lobby|terrace|club\s*house|clubhouse|amenity|mechanical|"
        r"parking|podium)\s+floor\b", re.IGNORECASE,
    )
    suffixless_re = re.compile(
        rf"\b(?:stilt|s|ground|g|basement|podium)\s*\+\s*"
        rf"(?P<count>\d+(?:\.\d+)?|{words})\b"
        r"(?!\s*(?:upper\s+)?floors?\b)", re.IGNORECASE,
    )
    for match in count_re.finditer(block_text):
        candidates.append((match.start(), match.end(), match.group(0), _number_value(match.group("count"))))
    for match in ordinal_re.finditer(block_text):
        candidates.append((match.start(), match.end(), match.group(0), 1))
    for match in named_re.finditer(block_text):
        candidates.append((match.start(), match.end(), match.group(0), 1))
    for match in suffixless_re.finditer(block_text):
        candidates.append((match.start(), match.end(), match.group("count"), _number_value(match.group("count"))))

    segments, total, occupied_until = [], 0, -1
    for start, end, segment, increment in sorted(candidates):
        if start < occupied_until:
            continue
        segments.append(re.sub(r"\s+", " ", segment.strip()))
        total += increment
        occupied_until = end
    return {
        "block": block_label,
        "base_level": base_level,
        "floor_segments": segments,
        "total_floors_above_base": total,
    }


def parse_floor_structures(project_details):
    if not isinstance(project_details, str) or not project_details.strip():
        return []
    results = []
    for block_label, block_text in _split_floor_blocks(project_details):
        parsed = _parse_floor_block(block_label, block_text)
        if parsed["base_level"] or parsed["floor_segments"]:
            results.append(parsed)
    return results


examples = [
    "Stilt Floor + 5 Upper Floors",
    "STILT+ FLOORS-TOTAL 190 UNITS",
    "STILT+ FIVE FLOOR",
    "S+5",
    "Basement Floor 1 & 2, Ground Floor + 2nd Floor + Lobby Floor + Terrace Floor",
    "Block 14: Combined Basement + Ground + 5 Upper Floors; Block 15: Combined Basement + Ground + 4 Upper Floors",
]
for number, example in enumerate(examples, start=1):
    print(f"{number}. {example}")
    print(parse_floor_structures(example))

1. Stilt Floor + 5 Upper Floors
[{'block': None, 'base_level': 'Stilt', 'floor_segments': ['5 Upper Floors'], 'total_floors_above_base': 5}]
2. STILT+ FLOORS-TOTAL 190 UNITS
[{'block': None, 'base_level': 'Stilt', 'floor_segments': [], 'total_floors_above_base': 0}]
3. STILT+ FIVE FLOOR
[{'block': None, 'base_level': 'Stilt', 'floor_segments': ['FIVE FLOOR'], 'total_floors_above_base': 5}]
4. S+5
[{'block': None, 'base_level': 'Stilt', 'floor_segments': ['5'], 'total_floors_above_base': 5}]
5. Basement Floor 1 & 2, Ground Floor + 2nd Floor + Lobby Floor + Terrace Floor
[{'block': None, 'base_level': 'Basement', 'floor_segments': ['2nd Floor', 'Lobby Floor', 'Terrace Floor'], 'total_floors_above_base': 3}]
6. Block 14: Combined Basement + Ground + 5 Upper Floors; Block 15: Combined Basement + Ground + 4 Upper Floors
[{'block': 'Block 14', 'base_level': 'Combined Basement + Ground', 'floor_segments': ['5 Upper Floors'], 'total_floors_above_base': 5}, {'block': 'Block 15', 'base_level': '

In [ ]:
def project_registration_no(source_file):
    """Convert a PDF filename stem such as TNRERA_1_BLG_0156_2026 to slash format."""
    filename = re.split(r"[\\/]", str(source_file))[-1]
    stem = re.sub(r"\.pdf$", "", filename, flags=re.IGNORECASE)
    return stem.replace("_", "/")

example_source_file = r"..\2026\TNRERA_1_BLG_0156_2026.pdf"
print(project_registration_no(example_source_file))

TNRERA/1/BLG/0156/2026


In [12]:
BASE_LEVEL_PATTERNS = [
    ("Combined Basement + Ground", r"\bcombined\s+(?:double\s+)?basement\s*\+\s*ground\b"),
    ("Ground", r"\bground\b|(?<!\w)g(?=\s*\+)"),
    ("Basement", r"\bbasement\b"),
    ("Stilt", r"\bstilt\b|(?<!\w)s(?=\s*\+)"),
    ("Podium", r"\bpodium\b"),
]

for number, example in enumerate(examples, start=1):
    print(f"{number}. {example}")
    print(parse_floor_structures(example))

1. Stilt Floor + 5 Upper Floors
[{'block': None, 'base_level': 'Stilt', 'floor_segments': ['5 Upper Floors'], 'total_floors_above_base': 5}]
2. STILT+ FLOORS-TOTAL 190 UNITS
[{'block': None, 'base_level': 'Stilt', 'floor_segments': [], 'total_floors_above_base': 0}]
3. STILT+ FIVE FLOOR
[{'block': None, 'base_level': 'Stilt', 'floor_segments': ['FIVE FLOOR'], 'total_floors_above_base': 5}]
4. S+5
[{'block': None, 'base_level': 'Stilt', 'floor_segments': ['5'], 'total_floors_above_base': 5}]
5. Basement Floor 1 & 2, Ground Floor + 2nd Floor + Lobby Floor + Terrace Floor
[{'block': None, 'base_level': 'Ground', 'floor_segments': ['2nd Floor', 'Lobby Floor', 'Terrace Floor'], 'total_floors_above_base': 3}]
6. Block 14: Combined Basement + Ground + 5 Upper Floors; Block 15: Combined Basement + Ground + 4 Upper Floors
[{'block': 'Block 14', 'base_level': 'Combined Basement + Ground', 'floor_segments': ['5 Upper Floors'], 'total_floors_above_base': 5}, {'block': 'Block 15', 'base_level': 'Co

In [ ]:
# Process all PDFs and save the extracted dataframe.
import glob

files = glob.glob(r"..\data_source\raw\TNRERA\*\*.pdf")
print(f"Found {len(files)} PDFs")

df, fail_df = run_batch(files)
df["Project_Registration_No"] = df["Source_File"].map(project_registration_no)

print(f"Parsed: {len(df)}   Failed to open/parse: {len(fail_df)}")
print(f"Rows with a missing core field: {(df['_missing_fields'] != '').sum()}")

df.to_csv(r"..\outputs\tnrera_projects_final.csv", index=False)
fail_df.to_csv(r"..\outputs\tnrera_failures.csv", index=False)
print("Saved: outputs/tnrera_projects_final.csv")

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Found 1030 PDFs


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

Parsed: 1030   Failed to open/parse: 0
Rows with a missing core field: 114


In [ ]:
print("Extracted rows:", len(df))
print("Failed PDF parses:", len(fail_df))
print("Rows with floor structures:", df["Floor_mentions"].map(bool).sum())
print("Output columns:", df.columns.tolist())
print("Saved extracted file: tnrera_projects_final.csv")

Extracted rows: 1030
Failed PDF parses: 0
Rows with floor structures: 518
Output columns: ['Source_File', 'Project_Name', 'Project_Details', 'Type_of_Building', 'Usage', 'Site_Extent_Sqm', 'Total_Dwelling_Units', 'Stage_of_Construction', 'Project_Completion_Date', 'Latitude', 'Longitude', 'Covered_Parking', 'Open_Parking', 'Visitor_Parking', 'Land_Cost', 'Construction_Cost', 'Other_Cost', 'Total_Project_Cost', 'FSI_LIG_Residential', 'FSI_Other_Residential', 'FSI_Commercial', 'Heights_m', 'Floor_mentions', 'Districts', '_missing_fields']
Saved extracted file: tnrera_projects.csv


In [14]:
df

,Source_File,Project_Name,Project_Details,Type_of_Building,Usage,Site_Extent_Sqm,Total_Dwelling_Units,Stage_of_Construction,Project_Completion_Date,Latitude,Longitude,Covered_Parking,Open_Parking,Visitor_Parking,Land_Cost,Construction_Cost,Other_Cost,Total_Project_Cost,FSI_LIG_Residential,FSI_Other_Residential,FSI_Commercial,Heights_m,Floor_mentions,Districts,_missing_fields
0,V:\Project\Project_Stick\data_source\raw\TNRER...,CRT Magilagam,"Ward-E,Block-21,TS No.2/3,Periyar Nagar,Erode ...",Non-High Rise Building (NHRB),Residential,1149.22,30,Under Construction,18/09/2028,11.335912,77.725072,22,6,3,24731214,111271043,16491620,152493877,0.00,3150.00,-,[],[],[Erode],
1,V:\Project\Project_Stick\data_source\raw\TNRER...,Mayflower East Gate,"Old S.F.No.418/2pt,New T.S.No.5/1,Block.07,War...",High Rise Building (HRB),Residential,2230.00,42,Under Construction,31/12/2026,11.2192,77.1134,27,15,42,80000000,230000000,15982769,325982769,-,6260.08,-,[],[],[Coimbatore],
2,V:\Project\Project_Stick\data_source\raw\TNRER...,CASAGRAND ALPINE,proposed construction of Residential Group Dev...,Non-High Rise Building (NHRB),Residential,5672.98,144,Not Yet Started,30/04/2029,110616.1,765959.1,97,2,2,12212576,203202738,23029644,238444958,-,11986.17,599.16,[],"[{'block': None, 'base_level': 'Stilt', 'floor...",[Coimbatore],
3,V:\Project\Project_Stick\data_source\raw\TNRER...,TIARA,"S.F.NO:370/3,CHINNAVEDAMPATTI,COIM BATORE-641049",Non-High Rise Building (NHRB),Residential,1133,22,Under Construction,31/12/2029,110530.7,765828.1,24,0,2,3659100,30000000,2917400,36576500,-,2913.67,-,[],[],[Coimbatore],
4,V:\Project\Project_Stick\data_source\raw\TNRER...,PNR MODERNA,"SF NO 129/4B, RANI GARDEN, 3RD STREET,SINGANAL...",Non-High Rise Building (NHRB),Residential,719.99,14,Under Construction,28/02/2027,11.006265,77.042583,16,0,0,26016000,43747218,11130881,80894099,-,1818.3,-,[],[],[Coimbatore],
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1025,V:\Project\Project_Stick\data_source\raw\TNRER...,VIJAYA HIGH VIEW,STILT + 05FLOORS 25DWELLING UNITS JARI KONDALA...,Non-High Rise Building (NHRB),Residential,1215.00,25,Under Construction,30/12/2027,11.635222,78.124639,23,5,2,981000,106519600,-,107500600,-,3156.92,-,[],"[{'block': None, 'base_level': 'Stilt', 'floor...",[Salem],
1026,V:\Project\Project_Stick\data_source\raw\TNRER...,SAKURA ENCLAVE,STILT+03 FLOORS APARTMENT BUILDING(12DWELLINGS),Non-High Rise Building (NHRB),Residential,920.92,12,Under Construction,30/12/2026,11.685556,78.133667,14,0,2,6500000,48729400,-,55229400,0.00,1751.37,-,[],"[{'block': None, 'base_level': 'Stilt', 'floor...",[Salem],
1027,V:\Project\Project_Stick\data_source\raw\TNRER...,METROPLOE - PUSHPA VIHAR,Apartment Building,Non-High Rise Building (NHRB),Residential,617.20,9,Under Construction,24/09/2030,11.676617,78.144727,12,0,2,7307500,40848000,-,48155500,0.00,1564.97,-,[],[],[Salem],
1028,V:\Project\Project_Stick\data_source\raw\TNRER...,Square Ball,Proposed Construction of Stilt + 5 Floors Resi...,Non-High Rise Building (NHRB),Residential,992.82,23,Not Yet Started,14/05/2033,11.68551,78.11725,21,2,3,17700000,55500000,-,73200000,0.00,2581.32,-,[],"[{'block': None, 'base_level': 'Stilt', 'floor...",[Salem],


In [15]:
empty_floor_mentions = df["Floor_mentions"].apply(lambda value: value == [])
rows_without_floor_mentions = df.loc[empty_floor_mentions]

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

print("Rows with Floor_mentions == []:", len(rows_without_floor_mentions))
display(rows_without_floor_mentions)

Rows with Floor_mentions == []: 512


,Source_File,Project_Name,Project_Details,Type_of_Building,Usage,Site_Extent_Sqm,Total_Dwelling_Units,Stage_of_Construction,Project_Completion_Date,Latitude,Longitude,Covered_Parking,Open_Parking,Visitor_Parking,Land_Cost,Construction_Cost,Other_Cost,Total_Project_Cost,FSI_LIG_Residential,FSI_Other_Residential,FSI_Commercial,Heights_m,Floor_mentions,Districts,_missing_fields
0,V:\Project\Project_Stick\data_source\raw\TNRERA\2024\TN_10_Building_0519_2024.pdf,CRT Magilagam,"Ward-E,Block-21,TS No.2/3,Periyar Nagar,Erode Tk & Dt.",Non-High Rise Building (NHRB),Residential,1149.22,30,Under Construction,18/09/2028,11.335912,77.725072,22,6,3,24731214,111271043,16491620,152493877,0.00,3150.00,-,[],[],[Erode],
1,V:\Project\Project_Stick\data_source\raw\TNRERA\2024\TN_11_Building_0301_2024.pdf,Mayflower East Gate,"Old S.F.No.418/2pt,New T.S.No.5/1,Block.07,Ward No.Z(26) of Vilankurichi Village,Coimbatore District",High Rise Building (HRB),Residential,2230.00,42,Under Construction,31/12/2026,11.2192,77.1134,27,15,42,80000000,230000000,15982769,325982769,-,6260.08,-,[],[],[Coimbatore],
3,V:\Project\Project_Stick\data_source\raw\TNRERA\2024\TN_11_Building_0341_2024.pdf,TIARA,"S.F.NO:370/3,CHINNAVEDAMPATTI,COIM BATORE-641049",Non-High Rise Building (NHRB),Residential,1133,22,Under Construction,31/12/2029,110530.7,765828.1,24,0,2,3659100,30000000,2917400,36576500,-,2913.67,-,[],[],[Coimbatore],
4,V:\Project\Project_Stick\data_source\raw\TNRERA\2024\TN_11_Building_0344_2024.pdf,PNR MODERNA,"SF NO 129/4B, RANI GARDEN, 3RD STREET,SINGANALLUR VILLAGE,COIMBATORE-641016",Non-High Rise Building (NHRB),Residential,719.99,14,Under Construction,28/02/2027,11.006265,77.042583,16,0,0,26016000,43747218,11130881,80894099,-,1818.3,-,[],[],[Coimbatore],
6,V:\Project\Project_Stick\data_source\raw\TNRERA\2024\TN_11_Building_0353_2024.pdf,ADHYA,"Adhya, SF.NO:577/1B2A, Chinnavedampatti Village, Coimbatore North Taluk, Coimbatore Corporation, Coimbatore District.",Non-High Rise Building (NHRB),Residential,5046.84,200,Not Yet Started,01/01/2026,11.04304,76.59039,95,0,13,54450000,835820000,3000000,893270000,-,14025.1,-,[],[],[Coimbatore],
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1018,V:\Project\Project_Stick\data_source\raw\TNRERA\2026\TNRERA_35_BLG_0271_2026.pdf,Enchante Apartments Phase 2,2 & 3 BHK RESIDENTIAL APARTMENTS,High Rise Building (HRB),Residential,9792,252,Not Yet Started,31/12/2030,125008.8,801235.0,85,97,11,39559000,1190241000,-,1229800000,0.00,26811.12,-,[],[],[Chengalpattu],
1020,V:\Project\Project_Stick\data_source\raw\TNRERA\2026\TNRERA_3_BLG_0246_2026.pdf,JAYALAKSHMI APARTMENTS,"S.F.No: 495/34,495/ 34A,495/46,495/46A. with site extent: 950.00 Sq.m, FSI Area:1939.20 Sq.m, Layout No: 316/ 2018.Vadakuthu Village Plot No-U, Kurinjipadi Taluk, Cuddalore District,Tamilnadu State.",High Rise Building (HRB),Residential,950.00,20,Not Yet Started,31/12/2029,11.625198,79.551704,17,0,2,45100000,81500000,-,126600000,0.00,1939.20,-,[],[],[Cuddalore],
1021,V:\Project\Project_Stick\data_source\raw\TNRERA\2026\TNRERA_5_BLG_0030_2026.pdf,RESIDENTIAL HOUSE,RESIDENTIAL APARTMENT,Non-High Rise Building (NHRB),Residential,646.60,15,Under Construction,31/05/2029,12.952803,79.140012,7,4,1,45100000,41000000,-,86100000,0.00,1288.85,-,[],[],[Vellore],
1023,V:\Project\Project_Stick\data_source\raw\TNRERA\2026\TNRERA_5_BLG_0264_2026.pdf,SM RAJ TOWERS,"T.S.No.142[old S.F.No.153(Pt)], Approved Layout No. 89/49, Plot No. A2/88 of Kalinjur Village, Vellore City Municipal Corporation, Katpadi Taluk, Vellore District",Non-High Rise Building (NHRB),Residential,646.59,15,Not Yet Started,04/05/2031,12.956000,79.139962,14,0,1,30000000,40000000,-,70000000,0.00,1288.65,-,[],[],[Vellore],


In [ ]:
rows_without_floor_mentions.to_csv(
    r"..\outputs\floor_mentions_empty_rows.csv",
    index=False,
)
print("Exported rows:", len(rows_without_floor_mentions))
print("Saved file: outputs/floor_mentions_empty_rows.csv")

Exported rows: 512
Saved file: src/floor_mentions_empty_rows.csv


In [20]:
nan_profile = pd.DataFrame({
    "column": df.columns,
    "dtype": [df[column].dtype for column in df.columns],
    "total_rows": len(df),
    "non_nan_count": [df[column].notna().sum() for column in df.columns],
    "nan_count": [df[column].isna().sum() for column in df.columns],
})
nan_profile["nan_percentage"] = (
    nan_profile["nan_count"] / nan_profile["total_rows"] * 100
).round(2)

display(nan_profile)

Saved file: src/nan_profile.csv


,column,dtype,total_rows,non_nan_count,nan_count,nan_percentage
0,Source_File,object,1030,1030,0,0.00
1,Project_Name,object,1030,1027,3,0.29
2,Project_Details,object,1030,1027,3,0.29
3,Type_of_Building,object,1030,1027,3,0.29
4,Usage,object,1030,1027,3,0.29
5,Site_Extent_Sqm,object,1030,1027,3,0.29
6,Total_Dwelling_Units,object,1030,943,87,8.45
7,Stage_of_Construction,object,1030,1027,3,0.29
8,Project_Completion_Date,object,1030,979,51,4.95
9,Latitude,object,1030,1027,3,0.29


# Final CSV file for further working

In [ ]:
df["Project_Registration_No"] = df["Source_File"].map(project_registration_no)

df.to_csv(
    r"..\outputs\tnrera_projects_final.csv",
    index=False,
)
print("Saved final extracted dataframe:", len(df), "rows")
print("Saved file: outputs/tnrera_projects_final.csv")
print(
    df.loc[
        df["Project_Registration_No"] == "TNRERA/1/BLG/0156/2026",
        ["Source_File", "Project_Registration_No"],
    ]
)

Saved final extracted dataframe: 1030 rows
Saved file: src/tnrera_projects_final.csv
                                                                         Source_File  \
824  V:\Project\Project_Stick\data_source\raw\TNRERA\2026\TNRERA_1_BLG_0156_2026.pdf   

    Project_Registration_No  
824  TNRERA/1/BLG/0156/2026  
